# TP2 : Classification Chat / Chien / Oiseau (EfficientNetB0)

**Avant de lancer (panneau de droite, Session options) :**
1. Ajouter le dataset : *Add Input* puis chercher `high-resolution-catdogbird-image-dataset-13000` (mahmoudnoor)
2. Activer **Internet** (pour télécharger les poids EfficientNetB0)
3. Activer le **GPU** (Accelerator)
4. Lancer avec **Run All**, sans cliquer sur Stop

## 1. Bibliothèques

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import warnings
warnings.filterwarnings("ignore")

print("Toutes les bibliothèques sont chargées !")
print("TensorFlow :", tf.__version__)
print("GPU disponible :", tf.config.list_physical_devices("GPU"))

## 2. Vérifier le dataset

In [ ]:
base_dir = "/kaggle/input/datasets/mahmoudnoor/high-resolution-catdogbird-image-dataset-13000"

print("Dataset disponible :", os.path.exists(base_dir))
print("Dossiers disponibles :", os.listdir(base_dir))

## 3. Création du tableau des images

In [ ]:
filepaths = []
labels = []

for category in ["bird", "cat", "dog"]:
    folder = os.path.join(base_dir, category, category)
    for filename in os.listdir(folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            filepaths.append(os.path.join(folder, filename))
            labels.append(category)

dataset = pd.DataFrame({"filepaths": filepaths, "labels": labels})

print("Nombre total d'images :", len(dataset))
print("\nRépartition des catégories :")
print(dataset["labels"].value_counts())

## 4. Séparer les données (80 % entraînement, 12 % validation, 8 % test)

In [ ]:
train_ds, temp_ds = train_test_split(
    dataset, train_size=0.8, random_state=123, shuffle=True, stratify=dataset["labels"]
)
valid_ds, test_ds = train_test_split(
    temp_ds, train_size=0.6, random_state=123, shuffle=True, stratify=temp_ds["labels"]
)

print("Images d'entraînement :", len(train_ds))
print("Images de validation :", len(valid_ds))
print("Images de test :", len(test_ds))

## 5. Préparer les images pour EfficientNetB0

In [ ]:
batch_size = 16
img_size = (224, 224)

# Augmentation des images d'entraînement uniquement
tr_gen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
)
# Pas d'augmentation pour validation et test
ts_gen = ImageDataGenerator()

train_gen = tr_gen.flow_from_dataframe(
    train_ds, x_col="filepaths", y_col="labels", target_size=img_size,
    class_mode="categorical", color_mode="rgb", batch_size=batch_size,
    shuffle=True, seed=123,
)
valid_gen = ts_gen.flow_from_dataframe(
    valid_ds, x_col="filepaths", y_col="labels", target_size=img_size,
    class_mode="categorical", color_mode="rgb", batch_size=batch_size, shuffle=False,
)
test_gen = ts_gen.flow_from_dataframe(
    test_ds, x_col="filepaths", y_col="labels", target_size=img_size,
    class_mode="categorical", color_mode="rgb", batch_size=batch_size, shuffle=False,
)

classes = list(train_gen.class_indices.keys())   # ['bird', 'cat', 'dog']
print("Classes :", classes)

## 6. Visualiser quelques images

In [ ]:
images, labels_batch = next(train_gen)

plt.figure(figsize=(12, 12))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].astype("uint8"))
    plt.title(classes[np.argmax(labels_batch[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Construction du modèle

In [ ]:
img_shape = (224, 224, 3)

# EfficientNetB0 pré-entraîné sur ImageNet (couches gelées)
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=img_shape, pooling="max"
)
base_model.trainable = False

model = Sequential([
    base_model,
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(3, activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 8. Entraînement

In [ ]:
# EarlyStopping : arrête si la validation ne s'améliore plus, et garde les meilleurs poids
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
]

# Les images sont en haute résolution : les charger prend ~1 seconde par lot.
# On limite donc la taille d'une époque pour que l'entraînement dure ~20 minutes.
epochs = 4
history = model.fit(
    train_gen,
    epochs=epochs,
    steps_per_epoch=250,      # 250 lots x 16 images = 4000 images par époque
    validation_data=valid_gen,
    validation_steps=60,      # 60 lots x 16 images = 960 images de validation
    callbacks=callbacks,
    verbose=1,
)

## 9. Courbes d'apprentissage

In [ ]:
tr_acc, val_acc = history.history["accuracy"], history.history["val_accuracy"]
tr_loss, val_loss = history.history["loss"], history.history["val_loss"]
epochs_range = range(1, len(tr_acc) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, tr_acc, label="Entraînement")
plt.plot(epochs_range, val_acc, label="Validation")
plt.title("Précision (accuracy)")
plt.xlabel("Époque")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, tr_loss, label="Entraînement")
plt.plot(epochs_range, val_loss, label="Validation")
plt.title("Perte (loss)")
plt.xlabel("Époque")
plt.legend()

plt.tight_layout()
plt.show()

## 10. Évaluation sur le jeu de test

In [ ]:
# Prédictions sur le jeu de test (test_gen n'est pas mélangé : l'ordre est conservé)
preds = model.predict(test_gen, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

print(f"Précision finale - entraînement : {history.history['accuracy'][-1]:.2%}")
print(f"Précision finale - validation   : {history.history['val_accuracy'][-1]:.2%}")
print(f"Précision sur le jeu de test    : {np.mean(y_pred == y_true):.2%}")

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Matrice de confusion (jeu de test)")
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.tight_layout()
plt.show()

# Rapport de classification
print(classification_report(y_true, y_pred, target_names=classes))

## 11. Sauvegarde du modèle

In [ ]:
os.makedirs("saved_model", exist_ok=True)

model.save("saved_model/CatDogBird_model.keras")   # format utilisé par l'application Flask
model.save("saved_model/CatDogBird_model.h5")      # ancien format

# L'ordre des classes doit être identique dans l'application Flask
with open("saved_model/labels.json", "w", encoding="utf-8") as f:
    json.dump(classes, f)

# Version de TensorFlow à installer pour l'application
with open("saved_model/tf_version.txt", "w") as f:
    f.write(tf.__version__)

print("Modèle sauvegardé. Ordre des classes :", classes)
print("Version de TensorFlow à installer pour l'application :", tf.__version__)

## 12. Recharger le modèle et prédire une image

In [ ]:
from tensorflow.keras.models import load_model

already_trained_model = load_model("saved_model/CatDogBird_model.keras")

def load_image(image_path, show=True):
    """Charge une image au format attendu par le modèle : (1, 224, 224, 3).

    ATTENTION : on garde les pixels entre 0 et 255. EfficientNetB0 fait sa propre
    normalisation. Ne pas appliquer mobilenet_v2.preprocess_input ici.
    """
    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    img_array = tf.keras.utils.img_to_array(img)
    if show:
        plt.imshow(img_array.astype("uint8"))
        plt.axis("off")
        plt.show()
    return np.expand_dims(img_array, axis=0)

# On teste sur une image au hasard du jeu de test
sample = test_ds.sample(1, random_state=7).iloc[0]
pred = already_trained_model.predict(load_image(sample["filepaths"]))

print("Probabilités :", dict(zip(classes, pred[0].round(3))))
print("Classe réelle  :", sample["labels"])
print("Classe prédite :", classes[np.argmax(pred)], f"({np.max(pred):.1%})")

# Diagramme des probabilités
plt.figure(figsize=(6, 3.5))
plt.bar(classes, pred[0], color=["#4C72B0", "#55A868", "#C44E52"])
plt.ylim(0, 1)
plt.ylabel("Probabilité")
plt.title("Probabilités prédites pour cette image")
plt.show()

## 13. Plusieurs prédictions (vert = correct, rouge = erreur)

In [ ]:
samples = test_ds.sample(6, random_state=1)

plt.figure(figsize=(14, 9))
for k, (_, row) in enumerate(samples.iterrows()):
    batch = load_image(row["filepaths"], show=False)
    probs = already_trained_model.predict(batch, verbose=0)[0]
    pred_label = classes[int(np.argmax(probs))]
    plt.subplot(2, 3, k + 1)
    plt.imshow(batch[0].astype("uint8"))
    plt.axis("off")
    plt.title(
        f"Réel : {row['labels']}\nPrédit : {pred_label} ({probs.max():.0%})",
        color="green" if pred_label == row["labels"] else "red",
    )
plt.tight_layout()
plt.show()

## 14. Préparer le téléchargement du modèle

Après **Save Version → Save & Run All (Commit)**, télécharge `My_Model.zip` dans l'onglet **Output** de la version enregistrée.

In [ ]:
!zip -r My_Model.zip saved_model
!ls -la saved_model
!ls -la My_Model.zip